In [ ]:
# Installation for Azure Event Hub client library (run once)
# Event Hub supports Kafka protocol, so we use kafka-python for compatibility
!pip install kafka-python

In [ ]:
# Imports and basic configuration
import json
import uuid
import random
import time
from datetime import datetime, timedelta, timezone
from kafka import KafkaProducer
import numpy as np

# Configurable parameters
EVENTS_PER_SECOND = 10  # Rate of events per second
SIMULATION_HOURS = 8   # Duration to run the simulation in hours

In [ ]:
# Azure Event Hub configuration parameters
# Azure Event Hub is Kafka-compatible, allowing us to use the Kafka protocol

# EVENT_HUB_NAMESPACE: Your Event Hub namespace (format: <namespace>.servicebus.windows.net:9093)
EVENT_HUB_NAMESPACE = 'TBD'

# EVENT_HUB_NAME: The name of your Event Hub (topic in Kafka terminology)
EVENT_HUB_NAME = 'TBD'

# EVENT_HUB_CONNECTION_STRING: Get this from Azure Portal > Event Hubs > Shared access policies
EVENT_HUB_CONNECTION_STRING = 'TBD'

# Authentication configuration (required for Azure Event Hub)
SASL_USERNAME = '$ConnectionString'  # This value should remain as-is for Event Hub
SASL_PASSWORD = EVENT_HUB_CONNECTION_STRING
SASL_MECHANISM = 'PLAIN'
SECURITY_PROTOCOL = 'SASL_SSL'

In [ ]:
producer = None

def init_event_hub_producer(
    namespace, 
    sasl_username, 
    sasl_password, 
    sasl_mechanism='PLAIN', 
    security_protocol='SASL_SSL'
):
    """
    Initialize producer for Azure Event Hub using Kafka protocol.
    
    Args:
        namespace: Event Hub namespace with port (e.g., 'namespace.servicebus.windows.net:9093')
        sasl_username: Authentication username (use '$ConnectionString' for Event Hub)
        sasl_password: Event Hub connection string
        sasl_mechanism: SASL mechanism (default: 'PLAIN')
        security_protocol: Security protocol (default: 'SASL_SSL')
    """
    global producer
    kafka_config = {
        'bootstrap_servers': [namespace],
        'value_serializer': lambda v: json.dumps(v).encode('utf-8'),
        'acks': 'all',
        'retries': 3,
        'security_protocol': security_protocol,
        'sasl_mechanism': sasl_mechanism,
        'sasl_plain_username': sasl_username,
        'sasl_plain_password': sasl_password,
        'api_version_auto_timeout_ms': 10000,
        'request_timeout_ms': 30000,
        'connections_max_idle_ms': 540000
    }
    
    producer = KafkaProducer(**kafka_config)
    print(f"Azure Event Hub producer initialized to namespace: {namespace}")

def send_event_to_event_hub(event, event_hub_name):
    """
    Send event to Azure Event Hub.
    
    Args:
        event: Event data to send (will be JSON serialized)
        event_hub_name: Name of the Event Hub to send to
    """
    if producer is None:
        raise Exception("Event Hub producer not initialized")
    try:
        future = producer.send(event_hub_name, event)
        record_metadata = future.get(timeout=10)
        print(f"Event sent to Event Hub '{record_metadata.topic}' "
              f"partition {record_metadata.partition} offset {record_metadata.offset}")
        producer.flush()
    except Exception as e:
        print(f"Error sending event to Event Hub: {e}")

In [ ]:
from enum import Enum
import datetime
import time
import random

# class syntax
class EVENT_TYPE(Enum):
    voltage = 1
    reactive = 2
    current = 3

def generateVoltageEvent():
    event = {}
    event["UNS"] = "yourcompany/nld/helmond/building1/line1/ccSIM"
    event["deviceId"] = "ycCCSIM"
    event["timestamp"] = datetime.datetime.now().isoformat() + "Z"
    event["key"] = EVENT_TYPE.voltage.name   
    event["value"] = str(random.randint(22500, 23500) / 100) # "226.98"
    return event

def generateCurrentEvent():
    event = {}
    event["UNS"] = "yourcompany/nld/helmond/building1/line1/ccSIM"
    event["deviceId"] = "ycCCSIM"
    event["timestamp"] = datetime.datetime.now().isoformat() + "Z"
    event["key"] = EVENT_TYPE.current.name   
    if ( (datetime.datetime.now().minute >= 0) and (datetime.datetime.now().minute < 15) ):
        event["value"] = "0.02"
    if ( (datetime.datetime.now().minute >= 15) and (datetime.datetime.now().minute < 30) ):
        event["value"] = "0.15"
    if ( (datetime.datetime.now().minute >= 30) and (datetime.datetime.now().minute < 45) ):
        event["value"] = "0.03"
    if ( (datetime.datetime.now().minute >= 45) and (datetime.datetime.now().minute < 60) ):
        event["value"] = "0.16"
    return event

def generateReactiveEvent():
    event = {}
    event["UNS"] = "yourcompany/nld/helmond/building1/line1/ccSIM"
    event["deviceId"] = "ycCCSIM"
    event["timestamp"] = datetime.datetime.now().isoformat() + "Z"
    event["key"] = EVENT_TYPE.reactive.name   
    event["value"] = str(random.randint(-650, -550) / 100) # "-0.603"
    return event

In [ ]:
def generateEvents(
    event_hub_name,
    send_func=None, 
    print_events=True
):
    """
    Generate and send simulated energy meter events.
    
    Args:
        event_hub_name: Name of the Event Hub to send events to
        send_func: Function to call to send events (optional)
        print_events: Whether to print events to console (default: True)
    """
    try:
        while True:
            # Generate and send voltage event
            impressionEvent = generateVoltageEvent()    
            if print_events:
                print(json.dumps(impressionEvent))
        
            if send_func:
                send_func(impressionEvent, event_hub_name)
            time.sleep(0.2)

            # Generate and send current event
            impressionEvent = generateCurrentEvent()    
            if print_events:
                print(json.dumps(impressionEvent))
        
            if send_func:
                send_func(impressionEvent, event_hub_name)
            time.sleep(0.2)

            # Generate and send reactive event
            impressionEvent = generateReactiveEvent()    
            if print_events:
                print(json.dumps(impressionEvent))
        
            if send_func:
                send_func(impressionEvent, event_hub_name)
            time.sleep(0.2)

            time.sleep(1)
    except KeyboardInterrupt:
        if producer:
            producer.close()
        print("\nEvent generation stopped.")

In [ ]:
# Initialize Azure Event Hub Producer with authentication
init_event_hub_producer(
    EVENT_HUB_NAMESPACE, 
    SASL_USERNAME, 
    SASL_PASSWORD,
    SASL_MECHANISM,
    SECURITY_PROTOCOL
)

print(f"Starting event generation at {datetime.datetime.now()}")
print(f"Sending to Event Hub: {EVENT_HUB_NAME}")
print("Press Ctrl+C to stop\n")

# Start generating and sending events to Azure Event Hub
generateEvents(
    event_hub_name=EVENT_HUB_NAME,
    send_func=send_event_to_event_hub,
    print_events=True
)

print(f"Event generation completed at {datetime.datetime.now()}")